<a href="https://colab.research.google.com/github/CamiloJose90/Exporta_Co/blob/main/pipeline/exporta_co_pipeline_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exporta.co — Pipeline V3 (Colab, bajo consumo de RAM)
**Estrategia:** procesa un año a la vez, guarda Parquet inmediatamente y libera RAM antes de continuar.  
**Migración local:** cambiar `MODO = 'local'` en S1 y ajustar rutas.  
**Ejecucion rapida:** una vez generados los Parquet, comentar S5 y cargar directamente desde S6.

---
## S1 — Setup: instalacion, imports y rutas

In [32]:
# Instalaciones (solo necesarias en Colab)
import subprocess, sys
for pkg in ['pycountry', 'pyarrow']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import pandas as pd
import numpy as np
import json, zipfile, io, re, gc
from pathlib import Path
from collections import defaultdict
import pycountry

print(f'pandas {pd.__version__} | numpy {np.__version__}')

pandas 2.2.2 | numpy 2.0.2


In [33]:
MODO = 'colab'  # cambiar a 'local' para ejecucion local

if MODO == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')

    import subprocess
    REPO_DIR = Path('/content/Exporta_Co')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', 'https://github.com/CamiloJose90/Exporta_Co.git', str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull'])

    DRIVE_DIR  = Path('/content/drive/MyDrive/exporta_co')
    RAW_DIR    = DRIVE_DIR / 'raw'
    CLEAN_DIR  = DRIVE_DIR / 'clean'
    OUTPUT_DIR = REPO_DIR / 'data'

else:
    RAW_DIR    = Path(r'C:\Users\camil\OneDrive\Documentos\MIAD\Visualización y storytelling\Proyecto')
    CLEAN_DIR  = RAW_DIR / 'clean'
    OUTPUT_DIR = RAW_DIR / 'json_output'

CLEAN_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

assert RAW_DIR.exists(), f'RAW_DIR no encontrado: {RAW_DIR}'
print(f'RAW_DIR   : {RAW_DIR}')
print(f'CLEAN_DIR : {CLEAN_DIR}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RAW_DIR   : /content/drive/MyDrive/exporta_co/raw
CLEAN_DIR : /content/drive/MyDrive/exporta_co/clean
OUTPUT_DIR: /content/Exporta_Co/data


---
## S2 — Configuracion
Ajustar `AÑOS` para ampliar o reducir el rango. Si el DANE cambia columnas, editar los mapas aqui.

In [34]:
AÑOS = list(range(2011, 2026))

COLUMNAS_EXPO = {
    'ï»¿fech': 'fech', 'fech': 'fech',
    'cod_pai4': 'pais',       # ISO alfa-3 directo (USA, DEU...)
    'posar':    'subpartida', # posicion arancelaria HS
    'fobdol':   'fob_usd',
    'pnk':      'kg_neto',
}

COLUMNAS_IMPO = {
    'ï»¿fech': 'fech', 'fech': 'fech',
    'paisgen':  'pais',       # codigo numerico ISO 3166-1
    'naban':    'subpartida',
    'vacid':    'cif_usd',    # valor CIF dolares
    'pnk':      'kg_neto',
}

# Mes en espanol -> numero (extrae mes desde nombre de archivo IMPO)
MESES_ES = {
    'enero':1,'febrero':2,'marzo':3,'abril':4,'mayo':5,'junio':6,
    'julio':7,'agosto':8,'septiembre':9,'octubre':10,'noviembre':11,'diciembre':12,
}

# Sectores OMC desde capitulo HS (primeros 2 digitos de posicion arancelaria)
_SECTOR_BREAKS  = [0, 24, 27, 97]
_SECTOR_LABELS  = ['Agropecuarios, alimentos y bebidas',
                   'Combustibles e industrias extractivas',
                   'Manufacturas']

CODIGOS_PAIS_IMPO = {
    13:'AFG', 17:'ALB', 23:'DEU', 26:'ARM', 27:'ABW', 37:'AND', 40:'AGO',
    43:'ATG', 47:'ANT', 53:'SAU', 59:'DZA', 63:'ARG', 69:'AUS', 72:'AUT',
    74:'AZE', 77:'BHS', 80:'BHR', 81:'BGD', 83:'BRB', 87:'BEL', 88:'BLZ',
    90:'BMU', 91:'BLR', 93:'MMR', 97:'BOL', 101:'BWA', 105:'BRA', 108:'BRN',
    111:'BGR', 115:'BDI', 119:'BTN', 127:'CPV', 137:'CYM', 141:'KHM',
    145:'CMR', 149:'CAN', 169:'COL', 173:'COM', 177:'COG', 187:'PRK',
    190:'KOR', 193:'CIV', 196:'CRI', 198:'HRV', 199:'CUB', 203:'TCD',
    211:'CHL', 215:'CHN', 218:'TWN', 221:'CYP', 229:'BEN', 232:'DNK',
    235:'DMA', 239:'ECU', 240:'EGY', 242:'SLV', 243:'ERI', 244:'ARE',
    245:'ESP', 246:'SVK', 247:'SVN', 249:'USA', 251:'EST', 253:'ETH',
    267:'PHL', 271:'FIN', 275:'FRA', 281:'GAB', 285:'GMB', 287:'GEO',
    289:'GHA', 297:'GRD', 301:'GRC', 317:'GTM', 329:'GIN', 331:'GNQ',
    334:'GNB', 337:'GUY', 341:'HTI', 345:'HND', 351:'HKG', 355:'HUN',
    361:'IND', 365:'IDN', 369:'IRQ', 372:'IRN', 375:'IRL', 379:'ISL',
    383:'ISR', 386:'ITA', 391:'JAM', 399:'JPN', 403:'JOR', 406:'KAZ',
    410:'KEN', 412:'KGZ', 413:'KWT', 420:'LAO', 426:'LSO', 429:'LVA',
    431:'LBN', 434:'LBR', 438:'LBY', 440:'LIE', 443:'LTU', 445:'LUX',
    447:'MAC', 450:'MDG', 455:'MYS', 458:'MWI', 461:'MDV', 464:'MLI',
    467:'MLT', 474:'MAR', 477:'MTQ', 485:'MUS', 488:'MRT', 493:'MEX',
    496:'MDA', 497:'MNG', 498:'MCO', 505:'MOZ', 507:'NAM', 517:'NPL',
    521:'NIC', 525:'NER', 528:'NGA', 538:'NOR', 542:'NCL', 545:'PNG',
    548:'NZL', 551:'VUT', 556:'OMN', 573:'NLD', 576:'PAK', 580:'PAN',
    586:'PRY', 589:'PER', 603:'POL', 607:'PRT', 611:'PRI', 618:'QAT',
    628:'GBR', 640:'CAF', 644:'CZE', 647:'DOM', 660:'REU', 665:'ZWE',
    670:'ROU', 675:'RWA', 676:'RUS', 685:'ESH', 687:'WSM', 695:'KNA',
    697:'SMR', 705:'VCT', 715:'LCA', 720:'STP', 728:'SEN', 729:'SRB',
    731:'SYC', 735:'SLE', 741:'SGP', 744:'SYR', 748:'SOM', 750:'LKA',
    756:'ZAF', 759:'SDN', 764:'SWE', 767:'CHE', 770:'SUR', 774:'TJK',
    776:'THA', 780:'TZA', 783:'DJI', 788:'TLS', 800:'TGO', 810:'TON',
    815:'TTO', 820:'TUN', 825:'TKM', 827:'TUR', 830:'UKR', 833:'UGA',
    845:'URY', 847:'UZB', 850:'VEN', 855:'VNM', 863:'VGB', 866:'VIR',
    870:'FJI', 880:'YEM', 888:'COD', 890:'ZMB',
    **{c: None for c in range(911, 930)}, 999: None,
}
print(f'Diccionario DANE cargado: {len(CODIGOS_PAIS_IMPO)} codigos')


print('Configuracion lista')

Diccionario DANE cargado: 214 codigos
Configuracion lista


---
## S3 — Funciones auxiliares

In [35]:
def normalizar_columnas(df, mapa):
    rename = {}
    for col in df.columns:
        clave = col.strip().lower().replace(' ', '_')
        if clave in mapa:      rename[col] = mapa[clave]
        elif col.strip() in mapa: rename[col] = mapa[col.strip()]
    return df.rename(columns=rename)


def limpiar_valor_monetario(serie):
    """Formato europeo: punto=miles, coma=decimal (ej: 4.885,70 -> 4885.70)."""
    return (serie.astype(str).str.strip()
            .str.replace('.', '', regex=False)
            .str.replace(',', '.', regex=False)
            .pipe(pd.to_numeric, errors='coerce')
            .astype('float32'))


def derivar_sector(serie_subpartida):
    """Sector OMC desde primeros 2 digitos de la posicion arancelaria (vectorizado)."""
    cap = pd.to_numeric(serie_subpartida.astype(str).str[:2], errors='coerce')
    return (pd.cut(cap, bins=_SECTOR_BREAKS, labels=_SECTOR_LABELS)
            .astype(str).replace('nan', 'Otros sectores').astype('category'))


def mes_desde_nombre(nombre_archivo):
    """Extrae numero de mes desde nombre del archivo CSV (ej: 'Agosto.csv' -> 8)."""
    return MESES_ES.get(Path(nombre_archivo).stem.lower().strip(), None)


def leer_csv_bytes(data_bytes):
    """Lee CSV desde bytes probando encodings y separadores comunes del DANE."""
    for encoding in ('latin-1', 'utf-8', 'utf-8-sig'):
        for sep in (',', ';', '\t'):
            try:
                df = pd.read_csv(io.BytesIO(data_bytes), sep=sep, encoding=encoding,
                                 low_memory=False, on_bad_lines='skip')
                if len(df.columns) > 2:
                    return df.dropna(how='all')
            except Exception:
                continue
    return None


_cache_iso3 = {}
def codigo_numerico_a_iso3(codigo):
    """Convierte codigo numerico ISO 3166-1 a ISO alfa-3 (con cache)."""
    if pd.isna(codigo): return None
    key = str(int(codigo)).zfill(3)
    if key not in _cache_iso3:
        try:    _cache_iso3[key] = pycountry.countries.get(numeric=key).alpha_3
        except: _cache_iso3[key] = None
    return _cache_iso3[key]


print('Funciones listas')

Funciones listas


---
## S4 — Descubrimiento de archivos
Verificar que todos los ZIPs esten correctamente ubicados antes de procesar.

In [36]:
PATRON = re.compile(r'^(expo|impo)_(\d{4})(?:_(\d))?\.zip$', re.IGNORECASE)
archivos = {'expo': defaultdict(list), 'impo': defaultdict(list)}

for f in sorted(RAW_DIR.glob('*.zip')):
    m = PATRON.match(f.name)
    if m:
        archivos[m.group(1).lower()][int(m.group(2))].append(f)

for tipo in ('expo', 'impo'):
    en_rango   = sorted(a for a in archivos[tipo] if a in AÑOS)
    faltantes  = [a for a in AÑOS if a not in archivos[tipo]]
    print(f'\n{tipo.upper()} — {len(en_rango)} años encontrados')
    for año in en_rango:
        partes = archivos[tipo][año]
        tag = f'[{len(partes)} partes]' if len(partes) > 1 else ''
        for p in partes: print(f'  {p.name} {tag}')
    if faltantes: print(f'  Faltantes: {faltantes}')


EXPO — 15 años encontrados
  Expo_2011.zip 
  Expo_2012.zip 
  Expo_2013.zip 
  Expo_2014.zip 
  Expo_2015.zip 
  Expo_2016.zip 
  Expo_2017.zip 
  Expo_2018.zip 
  Expo_2019.zip 
  Expo_2020.zip 
  Expo_2021.zip 
  Expo_2022.zip 
  Expo_2023.zip 
  Expo_2024.zip 
  Expo_2025.zip 

IMPO — 15 años encontrados
  Impo_2011.zip 
  Impo_2012.zip 
  Impo_2013.zip 
  Impo_2014.zip 
  Impo_2015.zip 
  Impo_2016.zip 
  Impo_2017.zip 
  Impo_2018.zip 
  Impo_2019.zip 
  Impo_2020.zip 
  Impo_2021_1.zip [2 partes]
  Impo_2021_2.zip [2 partes]
  Impo_2022_1.zip [2 partes]
  Impo_2022_2.zip [2 partes]
  Impo_2023.zip 
  Impo_2024.zip 
  Impo_2025_1.zip [2 partes]
  Impo_2025_2.zip [2 partes]


---
## S5 — Procesamiento año a año con liberacion de RAM
Cada año se procesa, limpia y guarda como Parquet de forma independiente.  
La RAM se libera despues de cada año con `gc.collect()`.  
**Para omitir este paso en ejecuciones posteriores:** comentar la llamada a `procesar_todo()` y cargar desde S6.

In [37]:
def procesar_zip_expo(ruta_zip):
    """Lee ZIP EXPO: ZIP anual > ZIPs mensuales > CSV."""
    frames = []
    try:
        with zipfile.ZipFile(ruta_zip, 'r') as zf:
            zips_mes = sorted(f for f in zf.namelist() if f.lower().endswith('.zip'))
            for nom in zips_mes:
                try:
                    with zipfile.ZipFile(io.BytesIO(zf.read(nom)), 'r') as zf_mes:
                        for csv_path in (f for f in zf_mes.namelist() if f.lower().endswith('.csv')):
                            df = leer_csv_bytes(zf_mes.read(csv_path))
                            if df is not None:
                                frames.append(normalizar_columnas(df, COLUMNAS_EXPO))
                except Exception as e:
                    print(f'    Advertencia {nom}: {e}')
    except zipfile.BadZipFile:
        print(f'    ZIP corrupto: {ruta_zip.name}')
    return frames


def procesar_zip_impo(ruta_zip):
    """
    Lee ZIP IMPO con dos casos:
    - Caso A (2011-2023, 2025): ZIP anual > ZIPs mensuales > CSV
    - Caso B (2024): ZIP anual > carpetas > CSV directos
    """
    frames = []
    try:
        with zipfile.ZipFile(ruta_zip, 'r') as zf:
            contenidos = zf.namelist()
            zips_mes   = sorted(f for f in contenidos if f.lower().endswith('.zip'))
            csvs_dir   = sorted(f for f in contenidos if f.lower().endswith('.csv'))

            if zips_mes:  # Caso A
                for nom in zips_mes:
                    try:
                        with zipfile.ZipFile(io.BytesIO(zf.read(nom)), 'r') as zf_mes:
                            for csv_path in (f for f in zf_mes.namelist() if f.lower().endswith('.csv')):
                                df = leer_csv_bytes(zf_mes.read(csv_path))
                                if df is not None:
                                    df = normalizar_columnas(df, COLUMNAS_IMPO)
                                    mes = mes_desde_nombre(csv_path)
                                    if mes: df['mes'] = mes
                                    frames.append(df)
                    except Exception as e:
                        print(f'    Advertencia {nom}: {e}')

            elif csvs_dir:  # Caso B
                for csv_path in csvs_dir:
                    try:
                        df = leer_csv_bytes(zf.read(csv_path))
                        if df is not None:
                            df = normalizar_columnas(df, COLUMNAS_IMPO)
                            mes = mes_desde_nombre(csv_path)
                            if mes: df['mes'] = mes
                            frames.append(df)
                    except Exception as e:
                        print(f'    Advertencia {csv_path}: {e}')

    except zipfile.BadZipFile:
        print(f'    ZIP corrupto: {ruta_zip.name}')
    return frames


def limpiar_expo(df, año):
    fech = pd.to_numeric(df['fech'], errors='coerce')
    df['año'] = año
    df['mes'] = (fech % 100)  # float por ahora, cast seguro al final
    df['iso3']    = df['pais'].str.strip().str.upper().astype('category')
    df['fob_usd'] = limpiar_valor_monetario(df['fob_usd'])
    df['sector']  = derivar_sector(df['subpartida'])
    df = df[
        df['fob_usd'].notna() & (df['fob_usd'] > 0) &
        df['iso3'].notna() & (df['iso3'] != '') &
        df['mes'].between(1, 12)
    ][['año', 'mes', 'iso3', 'fob_usd', 'sector']].copy()
    df['mes'] = df['mes'].astype('uint8')  # cast solo tras filtrar NaN
    return df


def _resolver_iso3_impo(codigo):
    """Resuelve codigo pais IMPO: primero diccionario DANE, luego pycountry como fallback."""
    if pd.isna(codigo):
        return None
    c = int(codigo)
    if c in CODIGOS_PAIS_IMPO:          # codigos historicos DANE (249=USA, etc.)
        return CODIGOS_PAIS_IMPO[c]
    try:                                 # codigos ISO numericos (archivos recientes)
        return pycountry.countries.get(numeric=str(c).zfill(3)).alpha_3
    except Exception:
        return None

_cache_resolver = {}
def resolver_iso3_impo_cached(codigo):
    if pd.isna(codigo): return None
    key = int(codigo)
    if key not in _cache_resolver:
        _cache_resolver[key] = _resolver_iso3_impo(codigo)
    return _cache_resolver[key]


_cache_impo = {}
def _iso3_impo(codigo, usar_dane):
    if pd.isna(codigo): return None
    key = (int(codigo), usar_dane)
    if key in _cache_impo: return _cache_impo[key]
    c = int(codigo)
    if usar_dane:
        result = CODIGOS_PAIS_IMPO.get(c)  # None si no esta en diccionario
    else:
        try:    result = pycountry.countries.get(numeric=str(c).zfill(3)).alpha_3
        except: result = None
    _cache_impo[key] = result
    return result

def limpiar_impo(df, año):
    fech = pd.to_numeric(df['fech'], errors='coerce')
    df['año'] = año
    if 'mes' not in df.columns:
        df['mes'] = (fech % 100)
    df['mes'] = pd.to_numeric(df['mes'], errors='coerce')
    df['cif_usd'] = (df['cif_usd'].astype(str).str.strip()
                     .str.replace(',', '.', regex=False)
                     .pipe(pd.to_numeric, errors='coerce')
                     .astype('float32'))
    usar_dane = (año <= 2024)  # 2011-2024: codigos DANE | 2025+: ISO numerico
    codigo = pd.to_numeric(df['pais'], errors='coerce')
    df['iso3'] = codigo.map(lambda x: _iso3_impo(x, usar_dane)).astype('category')
    df = df[
        df['cif_usd'].notna() & (df['cif_usd'] > 0) &
        df['iso3'].notna() &
        df['mes'].between(1, 12)
    ][['año', 'mes', 'iso3', 'cif_usd']].copy()
    df['mes'] = df['mes'].astype('uint8')
    return df


def procesar_todo():
    for tipo, fn_zip, fn_limpiar in [
        ('expo', procesar_zip_expo, limpiar_expo),
        ('impo', procesar_zip_impo, limpiar_impo),
    ]:
        print(f'\nProcesando {tipo.upper()}...')
        for año in AÑOS:
            parquet_path = CLEAN_DIR / f'{tipo}_{año}.parquet'
            if parquet_path.exists():
                print(f'  {año}: ya existe, omitido')
                continue

            partes = archivos[tipo].get(año, [])
            if not partes:
                print(f'  {año}: ZIP no encontrado')
                continue

            frames = []
            for ruta in sorted(partes):
                frames.extend(fn_zip(ruta))

            if not frames:
                print(f'  {año}: sin datos legibles')
                continue

            df = pd.concat(frames, ignore_index=True)
            df = fn_limpiar(df, año)
            df.to_parquet(parquet_path, index=False)
            n = len(df)
            del df, frames
            gc.collect()
            print(f'  {año}: {n:,} filas guardadas')

procesar_todo()


Procesando EXPO...
  2011: ya existe, omitido
  2012: ya existe, omitido
  2013: ya existe, omitido
  2014: ya existe, omitido
  2015: ya existe, omitido
  2016: ya existe, omitido
  2017: ya existe, omitido
  2018: ya existe, omitido
  2019: ya existe, omitido
  2020: ya existe, omitido
  2021: ya existe, omitido
  2022: ya existe, omitido
  2023: ya existe, omitido
  2024: ya existe, omitido
  2025: ya existe, omitido

Procesando IMPO...
  2011: 2,651,466 filas guardadas
  2012: 2,896,186 filas guardadas
  2013: 3,006,480 filas guardadas
  2014: 3,123,528 filas guardadas
  2015: 2,965,059 filas guardadas
  2016: 2,905,489 filas guardadas
  2017: 3,111,733 filas guardadas
  2018: 2,769,889 filas guardadas
  2019: 3,199,310 filas guardadas
  2020: 2,852,491 filas guardadas
  2021: 2,998,269 filas guardadas
  2022: 3,571,712 filas guardadas
  2023: 2,638,829 filas guardadas
  2024: 2,923,418 filas guardadas
  2025: 2,348,494 filas guardadas


---
## S6 — Carga desde Parquet
Punto de entrada para ejecuciones posteriores cuando los Parquet ya existen.  
En primera ejecucion S5 los genera; aqui solo se cargan.

In [38]:
df_expo = pd.concat(
    [pd.read_parquet(f) for f in sorted(CLEAN_DIR.glob('expo_*.parquet'))],
    ignore_index=True
)
df_impo = pd.concat(
    [pd.read_parquet(f) for f in sorted(CLEAN_DIR.glob('impo_*.parquet'))],
    ignore_index=True
)
print(f'Expo: {len(df_expo):,} filas | {df_expo["año"].nunique()} años')
print(f'Impo: {len(df_impo):,} filas | {df_impo["año"].nunique()} años')

Expo: 6,835,866 filas | 15 años
Impo: 43,962,353 filas | 15 años


---
## S7 — Generacion de JSON para D3.js

In [39]:
# VIZ 1 — Mapa coropletico: FOB por pais destino y año
viz1 = (
    df_expo
    .groupby(['año', 'iso3'], as_index=False)['fob_usd']
    .sum()
    .assign(fob_musd=lambda d: (d['fob_usd'] / 1e6).round(2))
    .drop(columns='fob_usd')
    .sort_values(['año', 'fob_musd'], ascending=[True, False])
)
(OUTPUT_DIR / 'viz1_mapa_destinos.json').write_text(
    json.dumps(viz1.to_dict(orient='records'), ensure_ascii=False, indent=2),
    encoding='utf-8'
)
print(f'viz1: {len(viz1):,} registros | {viz1["iso3"].nunique()} paises')

viz1: 2,811 registros | 225 paises


In [40]:
# VIZ 2 — Balanza comercial mensual
expo_m = df_expo.groupby(['año','mes'])['fob_usd'].sum().reset_index(name='expo_usd')
impo_m = df_impo.groupby(['año','mes'])['cif_usd'].sum().reset_index(name='impo_usd')
viz2 = (
    pd.merge(expo_m, impo_m, on=['año','mes'], how='outer').fillna(0)
    .assign(
        periodo            = lambda d: d['año'].astype(str) + '-' + d['mes'].astype(str).str.zfill(2),
        exportaciones_musd = lambda d: (d['expo_usd'] / 1e6).round(2),
        importaciones_musd = lambda d: (d['impo_usd'] / 1e6).round(2),
        balanza_musd       = lambda d: ((d['expo_usd'] - d['impo_usd']) / 1e6).round(2),
    )
    .sort_values('periodo')
    [['periodo','año','mes','exportaciones_musd','importaciones_musd','balanza_musd']]
)
(OUTPUT_DIR / 'viz2_balanza_temporal.json').write_text(
    json.dumps(viz2.to_dict(orient='records'), ensure_ascii=False, indent=2),
    encoding='utf-8'
)
print(f'viz2: {len(viz2):,} periodos ({viz2["periodo"].min()} -> {viz2["periodo"].max()})')

viz2: 180 periodos (2011-01 -> 2025-12)


In [41]:
# VIZ 3 — Treemap sectores OMC
sector_anual = (
    df_expo.groupby(['año','sector'])['fob_usd'].sum().reset_index()
    .assign(fob_musd=lambda d: (d['fob_usd'] / 1e6).round(2))
)
total_año = sector_anual.groupby('año')['fob_musd'].transform('sum')
sector_anual['pct'] = (sector_anual['fob_musd'] / total_año * 100).round(1)

años_disp = sorted(sector_anual['año'].unique().tolist())
viz3 = {
    'años_disponibles': años_disp,
    'datos_por_año': {
        str(año): {
            'name': f'Exportaciones {año}',
            'children': [
                {'name': r['sector'], 'value': r['fob_musd'], 'pct': r['pct']}
                for _, r in g.sort_values('fob_musd', ascending=False).iterrows()
            ]
        }
        for año, g in sector_anual.groupby('año')
    }
}
(OUTPUT_DIR / 'viz3_treemap_sectores.json').write_text(
    json.dumps(viz3, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(f'viz3: {len(años_disp)} años | sectores: {sector_anual["sector"].nunique()}')

/tmp/ipython-input-427/666170470.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_expo.groupby(['año','sector'])['fob_usd'].sum().reset_index()


viz3: 15 años | sectores: 4


---
## S8 — Push a GitHub
Solo ejecutar en Colab. Requiere token de acceso personal en `GITHUB_TOKEN`.  
El token no debe quedar guardado en el notebook antes de hacer commit.

In [52]:
import subprocess
from pathlib import Path
import time

# DOCUMENTACION: PUSH V3.7 - SINCRONIZACION UNIVERSAL
# Resuelve el conflicto entre el guardado manual de Colab y el push del script.

if MODO == 'colab':
    from google.colab import userdata

    try:
        GITHUB_TOKEN = userdata.get('GIT_TOKEN')
    except Exception:
        print("Error: No se encontro el secreto GIT_TOKEN en el panel de la llave.")
        GITHUB_TOKEN = None

    if GITHUB_TOKEN:
        REPO_URL_AUTH = f'https://{GITHUB_TOKEN}@github.com/CamiloJose90/Exporta_Co.git'

        # Secuencia de comandos optimizada
        cmds = [
            ['git', '-C', str(REPO_DIR), 'config', 'user.email', 'pipeline@exporta.co'],
            ['git', '-C', str(REPO_DIR), 'config', 'user.name', 'Exporta Pipeline'],
            ['git', '-C', str(REPO_DIR), 'remote', 'set-url', 'origin', REPO_URL_AUTH],
            # 1. Traer cambios remotos (lo que guardaste manualmente)
            ['git', '-C', str(REPO_DIR), 'pull', 'origin', 'main', '--rebase'],
            # 2. Agregar TODOS los cambios detectados en la carpeta clonada
            ['git', '-C', str(REPO_DIR), 'add', '.'],
            # 3. Intentar commit
            ['git', '-C', str(REPO_DIR), 'commit', '-m', f'v3: sincronizacion de datos {time.strftime("%H:%M:%S")}'],
            # 4. Enviar a GitHub
            ['git', '-C', str(REPO_DIR), 'push', 'origin', 'main']
        ]

        print("Sincronizando con GitHub...")
        for cmd in cmds:
            r = subprocess.run(cmd, capture_output=True, text=True)

            # Manejo de excepciones comunes
            if r.returncode != 0:
                # Si no hay nada que commitear, es un estado valido, no un error
                if "nothing to commit" in r.stdout or "nothing to commit" in r.stderr:
                    print("Nota: No se detectaron cambios nuevos en los archivos.")
                    continue
                # Si el push falla por estar atrasado (aunque ya hicimos pull)
                if "rejected" in r.stderr:
                    print("Error: El push fue rechazado. Intenta guardar el notebook manualmente una vez mas.")
                    break

                print(f'Error en comando: {" ".join(cmd)}\n{r.stderr}')
                break
        else:
            print(f'Sincronizacion completada con exito: {time.strftime("%H:%M:%S")}')
    else:
        print("Error: GIT_TOKEN no configurado.")
else:
    print('Modo local activo.')

Sincronizando con GitHub...
Nota: No se detectaron cambios nuevos en los archivos.
Sincronizacion completada con exito: 23:32:04
